In [1]:
import argparse
import glob
import os
import random
import numpy as np
import torch
from tqdm import tqdm
from os.path import join as pjoin

Task1: generate one sequence of subgoal with reactree using their prompt

In [2]:
from src.mcts_src.environment.alfworld_env import AlfWorldEnv


In [3]:
# select a task
def select_problems_fix(num_problems):
    # 固定的train的sample
    train_path = '/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen'
    first_level_dirs = next(os.walk(train_path))[1]

    # 然后对于每个子文件夹，获取它的第一个子文件夹
    results = []
    for dir_name in first_level_dirs:
        base_path = pjoin(train_path, dir_name)
        # 获取这个目录下的所有子文件夹
        sub_dirs = next(os.walk(base_path))[1]
        if sub_dirs:  # 如果有子文件夹
            # 获取第一个子文件夹的完整路径
            first_sub_dir = pjoin(base_path, sub_dirs[0])
            results.append(first_sub_dir)

    # 如果您还需要在这些第一个子文件夹中查找 initial_state.pddl
    valid_problems = []
    for dir_path in results:
        pddl_files = glob.glob(pjoin(dir_path, "initial_state.pddl"))
        if pddl_files and "movable_recep" not in pddl_files[0]:
            valid_problems.extend(pddl_files)
    # valid_problems = [item for item in valid_problems if "pick_two_obj_and_place" in item]
    return valid_problems[0]

In [4]:
task = select_problems_fix(1)
# ['/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen/pick_and_place_simple-PepperShaker-None-Drawer-10/trial_T20190918_154326_823501/initial_state.pddl']

In [5]:
task

'/home/azureuser/vikas/Embodied-Agent-Planning/mcts_datagen/data/json_2.1.1/valid_unseen/pick_and_place_simple-PepperShaker-None-Drawer-10/trial_T20190918_154326_823501/initial_state.pddl'

In [5]:
alf_env = AlfWorldEnv(config_path = '/home/azureuser/vikas/ReAcTree/conf/base_config.yaml', task_file=os.path.dirname(task))


Initialize AlfredThorEnv...
Overall we have 0 games...
Evaluating with 0 games
Overall we have 0 games...
Evaluating with 0 games
Found path: /home/azureuser/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64
Mono path[0] = '/home/azureuser/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64_Data/Managed'
Mono config path = '/home/azureuser/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64_Data/Mono/etc'
Unable to preload the following plugins:
	ScreenSelector.so
Display 0 'NVIDIA VGX  32"': 1024x768 (primary device).
Display 1 'NVIDIA VGX  32"': 1024x768 (secondary device).
Logging to /home/azureuser/.config/unity3d/Allen Institute for Artificial Intelligence/AI2-Thor/Player.log


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No su

ThorEnv started.
Resetting ThorEnv
Task: put a peppershaker in drawer


/home/azureuser/vikas/Embodied-Agent-Planning/.newnenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
!hf auth login

In [ ]:
!hf auth whoami

In [12]:
import os, importlib
from dotenv import load_dotenv
load_dotenv()

import src.rm.rm_llm_agent as _agent_mod
importlib.reload(_agent_mod)
from src.rm.rm_llm_agent import LlmAgent

llm = LlmAgent(hf_token=os.getenv('hf_token'))

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


In [ ]:
def prompt_template(self, state, available_commands):
        from rm.rm_prompt import MCTS_PROMPT
        system = MCTS_PROMPT
        messages = []
        if "Welcome to TextWorld, ALFRED!" in state.obs:
            messages.append({"role": "user", "content": f"Init environment: {state.obs}\n The candidate actions are {available_commands}"})
        else:
            for i in range(len(state.obs_list)):
                if i == 0:
                    messages.append({"role": "user", "content": f"Init environment: {state.obs_list[i]}\n"})
                    messages.append({"role": "assistant", "content": f"{state.action_history[i]}"})
                    continue
                if i == len(state.obs_list) - 1:
                    messages.append({"role": "user", "content": f"Current observation: {state.obs_list[i]}\nThe candidate actions are {available_commands}"})
                else:
                    messages.append({"role": "user", "content": f"Current observation: {state.obs_list[i]}"})
                    messages.append({"role": "assistant", "content": f"{state.action_history[i]}"})
        return system, messages

## First Expansion: pass ALFWorld task to LLM with ReAcTree prompt

In [17]:
import re

# Pull obs and info stored on the env object after reset
init_obs = alf_env.init_obs
init_info = alf_env.init_info  # returned by wait_and_get_info()[2]

# Extract "Your task is to: ..." from the initial observation
task_match = re.search(r'Your task is to: (.+)', init_obs)
task_desc = task_match.group(1).rstrip('.').strip() if task_match else "complete the task"

# admissible_commands is batched: info['admissible_commands'][batch_idx]
try:
    admissible_commands = init_info['admissible_commands'][0]
except (KeyError, TypeError, IndexError):
    admissible_commands = []

print("Task          :", task_desc)
print("Init obs      :", init_obs[:300])
print("Available cmds:", admissible_commands[:5], "...")

Task          : put some peppershaker on drawer
Init obs      : -= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, you see a fridge 1, a cabinet 1, a countertop 1, a toaster 1, a coffeemachine 1, a countertop 2, a cabinet 2, a stoveburner 1, a stoveburner 2, a cabinet 3, a cabinet 4, a microwave 1, a countertop 3, a
Available cmds: ['go to fridge 1', 'go to cabinet 1', 'go to countertop 1', 'go to toaster 1', 'go to coffeemachine 1'] ...


In [18]:
# Initialize LLM agent (loads Llama-3-8B-Instruct with your HF token)
# llm = LlmAgent(hf_token=os.getenv('hf_token'))

# Ask for the first expansion of the root goal
response = llm.first_expand(task_desc, init_obs, admissible_commands)
print(response)

cannot load prompt


UnboundLocalError: local variable 'REACTREE_PROMPT' referenced before assignment

In [ ]:
import importlib, src.rm.rm_llm_agent as _m
importlib.reload(_m)
from src.rm.rm_llm_agent import LlmAgent

parsed = LlmAgent.parse_expand_response(response)
print("Control flow:", parsed['control_flow'])
print("Subgoals:")
for i, s in enumerate(parsed['conditions'], 1):
    print(f"  {i}. {s}")

In [ ]:
from types import SimpleNamespace
from src.reactree import ControlFlowNode, AgentNode

# Minimal cfg stub — only max_depth is checked by ControlFlowNode.run()
cfg = SimpleNamespace(llm_agent=SimpleNamespace(max_depth=6))

# Root AgentNode (depth 0) — represents the full task
task_type = task.split('/')[-3].split('-')[0]  # e.g. 'pick_and_place_simple'
root = AgentNode(cfg, {'nl_inst': task_desc, 'task_type': task_type},
                 depth=0, llm_agent=llm, env=alf_env)

# ControlFlowNode (depth 1) — wires the expansion control strategy
cf_node = ControlFlowNode(cfg, parsed['control_flow'], depth=root.depth + 1)
root.add_child(cf_node)

# One AgentNode per subgoal (depth 2)
for subgoal in parsed['conditions']:
    child = AgentNode(cfg, {'nl_inst': subgoal, 'task_type': task_type},
                      depth=root.depth + 2, llm_agent=llm, env=alf_env)
    cf_node.add_child(child)

# Print the resulting tree
print(f"[AgentNode      d=0] {root.content['nl_inst']}")
print(f"  [ControlFlowNode d=1] {cf_node.content}")
for c in cf_node.children:
    print(f"    [AgentNode    d=2] {c.content['nl_inst']}")